In [442]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel(r"D:\Projects\Data\Sleep_Work_Productivity_Surve.xlsx")

# clean age column
df['Enter your age ?']= df['Enter your age ?'].fillna(df['Enter your age ?'].median())

# rename the columns
df.rename(columns={
    'How would you rate your overall work productivity?': 'work_productivity',
    'How much do you think your sleep affects your productivity at work?': 'sleep_productivity_effect',
    'What factors do you believe impact your sleep quality? (Select all that apply)': 'sleep_factors',
    'How many breaks do you take during your workday when you are tired ?': 'break_nums',
    'How would you describe your energy levels throughout the workday?': 'energy_levels_per_workday',
    'What is your primary work environment?': 'primary_work_environment',
    'what is you gender ?': 'gender',
    'On a typical work night, how many hours of sleep do you get?': 'sleep_hours',
    
    'I wake up feeling refreshed and ready for work.':
        'wake_up_refreshed',

    'I have difficulty falling asleep the night before an important workday.':
        'difficulty_falling_asleep',

    'Enter your age ?':
        'age',

    'I wake up multiple times during the night and struggle to go back to sleep.':
        'wake_up_multiple_times',

    'On weekends or days off, my sleep schedule is very different from my workday schedule (e.g., sleeping in more than 2 hours later).':
        'weekend_sleep_schedule_difference',

    'I use electronic devices (phone, laptop, TV) in bed within 30 minutes of trying to sleep.':
        'device_use_before_sleep',

    'I find it hard to concentrate on work tasks for more than 15–20 minutes at a time.':
        'difficulty_concentrating',

    'I make careless mistakes (e.g., typos, miscalculations, forgetting steps) that I would not have made if better rested.':
        'careless_mistakes',

    'I often have to re-read emails, documents, or instructions because I lost focus.':
        'reread_due_to_focus_loss',

    'Learning a new software, process, or skill at work feels unusually difficult for me.':
        'difficulty_learning_new_skills',

    'I struggle to generate new ideas or creative solutions to problems.':
        'difficulty_generating_ideas',

    'I forget important tasks, deadlines, or details from meetings.':
        'forget_tasks_deadlines',

    'I feel irritable or short-tempered with colleagues or clients.':
        'irritable_with_colleagues',

    'Small setbacks or minor criticism feel overwhelming or upsetting.':
        'small_setbacks_overwhelming',

    'I avoid collaboration or group work because I feel mentally exhausted.':
        'avoid_collaboration',

    'I have difficulty controlling my emotions during stressful work situations (e.g., tight deadlines, difficult customers).':
        'difficulty_controlling_emotions',

    'I feel physically tired, sluggish, or heavy-eyed during work, even without heavy physical exertion.':
        'physically_tired_at_work',

    'In my job, I have had a close call, minor accident, or injury that I attribute to being too tired. If your job is sedentary, answer based on accidents like tripping, bumping into things, or spilling hot liquids.':
        'accident_due_to_tiredness',

    'Compared to when I am well-rested, I estimate my current work output is...':
        'current_work_output',

    'In the past month, how many work days did you feel you were “present but not productive” (getting less than half of your normal output) due to poor sleep?':
        'unproductive_workdays_due_to_sleep'

}, inplace=True)


# clean the names
df = df.replace(r'\n', ', ', regex=True)
df["sleep_factors"] = df["sleep_factors"].str.split(", ")


# Reverse the only positive-direction question
df['wake_up_refreshed_reverse'] = 6 - df['wake_up_refreshed']


# clean unproductive_workdays_due_to_sleep outlires
df.loc[df["unproductive_workdays_due_to_sleep"] > 23,
       "unproductive_workdays_due_to_sleep"] = np.nan

median = df['unproductive_workdays_due_to_sleep'].median().round(0)
df['unproductive_workdays_due_to_sleep'] = df['unproductive_workdays_due_to_sleep'].fillna(median)


# 1. Sleep Quality and Hygiene Index (SQLI)
# 5 questions × maximum score of 5 = 25
# Divide by 25 and multiply by 100 → maximum possible score = 100
df['sleep_quality_and_hygiene_index_(SQLI)'] = (
    (
        df['wake_up_refreshed_reverse']
        + df['difficulty_falling_asleep']
        + df['wake_up_multiple_times']
        + df['weekend_sleep_schedule_difference']
        + df['device_use_before_sleep']
    ) / 25
) * 100


# 2. Cognitive at-task Performance Index (CTPI)
# 6 questions × maximum score of 5 = 30
# Divide by 30 and multiply by 100 → maximum possible score = 100
df['Cognitive_at_task_perforamnce_index_(CTPI)'] = (
    (df['difficulty_concentrating']
     + df['careless_mistakes']
     + df['reread_due_to_focus_loss']
     + df['difficulty_learning_new_skills']
     + df['difficulty_generating_ideas']
     + df['forget_tasks_deadlines']) / 30
) * 100

df['Cognitive_at_task_perforamnce_index_(CTPI)'] = (
    df['Cognitive_at_task_perforamnce_index_(CTPI)'].round(2)
)


# 3. Emotional and Social Impact Index (ESII)
# 4 questions × maximum score of 5 = 20
# Divide by 20 and multiply by 100 → maximum possible score = 100
df['Emotional_and_social_impact_index(ESII)'] = (
    (df['irritable_with_colleagues']
     + df['small_setbacks_overwhelming']
     + df['avoid_collaboration']
     + df['difficulty_controlling_emotions']) / 20
) * 100


# 4. Physical and Safety Risk Index (PSRI)
# 2 questions × maximum score of 5 = 10
# Divide by 10 and multiply by 100 → maximum possible score = 100
df['Physical_and_safety_risk_index(PSRI)'] = (
    (df['physically_tired_at_work']
     + df['accident_due_to_tiredness']) / 10
) * 100


# 5. Presentative Index (PI)
output_loss_map = {
    'About the same or better': 0.00,
    '20–40% less': 0.30,
    '40–60% less': 0.50,
    '60–80% less': 0.70,
    '80–100% less':1.00
}

df['productivity_loss_fraction'] = (
    df['current_work_output'].map(output_loss_map)
)

df['Presentative_index(PI)'] = (
    (df['unproductive_workdays_due_to_sleep'] / 23)
    * df['productivity_loss_fraction']
    * 100
)

df['Presentative_index(PI)'] = (
    df['Presentative_index(PI)'].round(2)
)


# 6. Overall Sleep Productivity Impact Index (SPI)
# Each component has a maximum of 100.
# Therefore, their total maximum = 400.
# Dividing by 4 gives an average with a maximum of 100.
df['Overall_sleep_Productivity_Impact_index(SPI)'] = (
    df['Cognitive_at_task_perforamnce_index_(CTPI)']
    + df['Emotional_and_social_impact_index(ESII)']
    + df['Physical_and_safety_risk_index(PSRI)']
    + df['Presentative_index(PI)']
) / 4

## **The Qs We Are Going To Answer, Using ML Models**

### ML 1 — SQLI
- Can we predict sleep quality and hygiene from workers' demographic, work, and productivity characteristics, and which characteristics are the most important predictors?

### ML 2 — SPI
- Can we predict overall sleep-related productivity impact from workers' sleep, work, and demographic characteristics, and which - - - characteristics are the most important predictors?

#### --> first of all we should split our data into 3 parts:
#### 1) df1 (the df what we wanna encode)
#### 2) df2 (the df what we will extract variables from)
#### 3) df3 (our lables)

In [443]:
df1 = df.loc[:, 'Submission Date':'sleep_hours']
df1['current_work_output']= df['current_work_output']
df1 = df1.drop(columns='sleep_factors')

In [444]:
df2 = df.loc[:,'wake_up_refreshed':].drop(columns=['Overall_sleep_Productivity_Impact_index(SPI)', 'sleep_quality_and_hygiene_index_(SQLI)', 'current_work_output'])

In [445]:
df3 = df[['Overall_sleep_Productivity_Impact_index(SPI)', 'sleep_quality_and_hygiene_index_(SQLI)']]

In [446]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Submission Date            127 non-null    str  
 1   work_productivity          127 non-null    str  
 2   sleep_productivity_effect  127 non-null    str  
 3   break_nums                 122 non-null    str  
 4   energy_levels_per_workday  127 non-null    str  
 5   primary_work_environment   125 non-null    str  
 6   gender                     127 non-null    str  
 7   sleep_hours                127 non-null    str  
 8   current_work_output        127 non-null    str  
dtypes: str(9)
memory usage: 9.1 KB


In [447]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 25 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   wake_up_refreshed                           127 non-null    int64  
 1   difficulty_falling_asleep                   127 non-null    int64  
 2   age                                         127 non-null    float64
 3   wake_up_multiple_times                      127 non-null    int64  
 4   weekend_sleep_schedule_difference           127 non-null    int64  
 5   device_use_before_sleep                     127 non-null    int64  
 6   difficulty_concentrating                    127 non-null    int64  
 7   careless_mistakes                           127 non-null    int64  
 8   reread_due_to_focus_loss                    127 non-null    int64  
 9   difficulty_learning_new_skills              127 non-null    int64  
 10  difficulty_generating_ide

In [448]:
df3.info()

<class 'pandas.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 2 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Overall_sleep_Productivity_Impact_index(SPI)  127 non-null    float64
 1   sleep_quality_and_hygiene_index_(SQLI)        127 non-null    float64
dtypes: float64(2)
memory usage: 2.1 KB


In [449]:
categorical_cols = [
    'work_productivity',
    'sleep_productivity_effect',
    'break_nums',
    'energy_levels_per_workday',
    'primary_work_environment',
    'gender',
    'sleep_hours',
    'current_work_output'
]

for col in categorical_cols:
    print("\n" + "="*50)
    print(col)
    print(df1[col].value_counts(dropna=False))


work_productivity
work_productivity
Moderate     64
High         35
Low          16
Very high    12
Name: count, dtype: int64

sleep_productivity_effect
sleep_productivity_effect
Significantly    59
Moderately       28
Extremely        28
A little          8
Not at all        4
Name: count, dtype: int64

break_nums
break_nums
3-4          43
1-2          43
5 or more    36
NaN           5
Name: count, dtype: int64

energy_levels_per_workday
energy_levels_per_workday
Moderate     84
High         24
Low          14
Very high     3
Very low      2
Name: count, dtype: int64

primary_work_environment
primary_work_environment
Hybrid        50
Office        44
Remote        25
Field work     6
NaN            2
Name: count, dtype: int64

gender
gender
Female               79
Male                 47
Prefer not to say     1
Name: count, dtype: int64

sleep_hours
sleep_hours
6–7 hrs            34
5–6 hrs            33
7–8 hrs            33
Less than 5 hrs    16
More than 8 hrs    11
Name: count,

In [450]:
#encode the variables of df1
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

encoded = encoder.fit_transform(
    df1[['primary_work_environment', 'gender']]
)

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(
        ['primary_work_environment', 'gender']
    ),
    index=df1.index
)

df1 = pd.concat([df1, encoded_df], axis=1)


work_productivity_map = {
    'Low': 0,
    'Moderate': 1,
    'High': 2,
    'Very high': 3
}

df1['work_productivity_encoded'] = (
    df1['work_productivity'].map(work_productivity_map)
)


sleep_effect_map = {
    'Not at all': 0,
    'A little': 1,
    'Moderately': 2,
    'Significantly': 3,
    'Extremely': 4
}

df1['sleep_productivity_effect_encoded'] = (
    df1['sleep_productivity_effect'].map(sleep_effect_map)
)


break_map = {
    '1-2': 1,
    '3-4': 2,
    '5 or more': 3
}

df1['break_nums_encoded'] = (
    df1['break_nums'].map(break_map)
)

break_nums_encoded_median = df1['break_nums_encoded'].median()
df1['break_nums_encoded'] = df1['break_nums_encoded'].fillna(break_nums_encoded_median)


energy_map = {
    'Very low': 0,
    'Low': 1,
    'Moderate': 2,
    'High': 3,
    'Very high': 4
}

df1['energy_encoded'] = (
    df1['energy_levels_per_workday'].map(energy_map)
)


sleep_hours_map = {
    'Less than 5 hrs': 0,
    '5–6 hrs': 1,
    '6–7 hrs': 2,
    '7–8 hrs': 3,
    'More than 8 hrs': 4
}

df1['sleep_hours_encoded'] = (
    df1['sleep_hours'].map(sleep_hours_map)
)


output_loss_map = {
    'About the same or better': 0.00,
    '20–40% less': 0.30,
    '40–60% less': 0.50,
    '60–80% less': 0.70,
    '80–100% less': 1.00
}

In [451]:
df1_encoded = df1.iloc[:, 9:]
df3_SQLI = df3['sleep_quality_and_hygiene_index_(SQLI)']
df3_SPI = df3['Overall_sleep_Productivity_Impact_index(SPI)']

In [452]:
df_SQLI = pd.concat([df1_encoded, df2, df3_SQLI], axis=1)

In [453]:
df_SPI = pd.concat([df1_encoded, df2, df3_SPI], axis=1)

In [454]:
df_SQLI.info()

<class 'pandas.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 39 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   primary_work_environment_Field work         127 non-null    float64
 1   primary_work_environment_Hybrid             127 non-null    float64
 2   primary_work_environment_Office             127 non-null    float64
 3   primary_work_environment_Remote             127 non-null    float64
 4   primary_work_environment_nan                127 non-null    float64
 5   gender_Female                               127 non-null    float64
 6   gender_Male                                 127 non-null    float64
 7   gender_Prefer not to say                    127 non-null    float64
 8   work_productivity_encoded                   127 non-null    int64  
 9   sleep_productivity_effect_encoded           127 non-null    int64  
 10  break_nums_encoded       

In [455]:
df_SPI.info()

<class 'pandas.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 39 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   primary_work_environment_Field work           127 non-null    float64
 1   primary_work_environment_Hybrid               127 non-null    float64
 2   primary_work_environment_Office               127 non-null    float64
 3   primary_work_environment_Remote               127 non-null    float64
 4   primary_work_environment_nan                  127 non-null    float64
 5   gender_Female                                 127 non-null    float64
 6   gender_Male                                   127 non-null    float64
 7   gender_Prefer not to say                      127 non-null    float64
 8   work_productivity_encoded                     127 non-null    int64  
 9   sleep_productivity_effect_encoded             127 non-null    int64  
 10  b

In [456]:
X1 = df_SQLI.drop(columns=[
    'sleep_quality_and_hygiene_index_(SQLI)',
    'wake_up_refreshed',
    'wake_up_refreshed_reverse',
    'difficulty_falling_asleep',
    'wake_up_multiple_times',
    'weekend_sleep_schedule_difference',
    'device_use_before_sleep', 
])

y1 = df_SQLI['sleep_quality_and_hygiene_index_(SQLI)']

In [457]:
X2 = df_SPI.drop(columns=[
    'Overall_sleep_Productivity_Impact_index(SPI)',
    'Cognitive_at_task_perforamnce_index_(CTPI)',
    'Emotional_and_social_impact_index(ESII)',
    'Physical_and_safety_risk_index(PSRI)',
    'Presentative_index(PI)'
])

y2 = df_SPI['Overall_sleep_Productivity_Impact_index(SPI)']

In [458]:
from sklearn.model_selection import train_test_split

# ML1 splitted
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1,
    y1,
    test_size=0.2,
    random_state=42
)

# ML2 splitted
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2,
    y2,
    test_size=0.2,
    random_state=42
)

In [459]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)

In [460]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

In [461]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(
    random_state=42
)

# **ML1 for SQLI**

In [462]:
ridge.fit(X1_train, y1_train)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [463]:
rf.fit(X1_train, y1_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [464]:
gb.fit(X1_train, y1_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf

In [465]:
y1_pred_ridge = ridge.predict(X1_test)
y1_pred_rf = rf.predict(X1_test)
y1_pred_gb = gb.predict(X1_test)

In [466]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ridge_mae = mean_absolute_error(y1_test, y1_pred_ridge)
ridge_rmse = np.sqrt(mean_squared_error(y1_test, y1_pred_ridge))
ridge_r2 = r2_score(y1_test, y1_pred_ridge)

In [467]:
rf_mae = mean_absolute_error(y1_test, y1_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y1_test, y1_pred_rf))
rf_r2 = r2_score(y1_test, y1_pred_rf)

In [468]:
gb_mae = mean_absolute_error(y1_test, y1_pred_gb)
gb_rmse = np.sqrt(mean_squared_error(y1_test, y1_pred_gb))
gb_r2 = r2_score(y1_test, y1_pred_gb)

In [469]:
results_sqli = pd.DataFrame({
    "Model": ["Ridge", "Random Forest", "Gradient Boosting"],
    "MAE": [ridge_mae, rf_mae, gb_mae],
    "RMSE": [ridge_rmse, rf_rmse, gb_rmse],
    "R²": [ridge_r2, rf_r2, gb_r2]
})

results_sqli.round(3)

,Model,MAE,RMSE,R²
0,Ridge,9.529,12.850,-0.162
1,Random Forest,8.986,11.905,0.003
2,Gradient Boosting,9.001,12.452,-0.091


In [470]:
best_mae_model = results_sqli.loc[results_sqli["MAE"].idxmin(), "Model"]
best_rmse_model = results_sqli.loc[results_sqli["RMSE"].idxmin(), "Model"]
best_r2_model = results_sqli.loc[results_sqli["R²"].idxmax(), "Model"]

print("Best MAE model:", best_mae_model)
print("Best RMSE model:", best_rmse_model)
print("Best R² model:", best_r2_model)

Best MAE model: Random Forest
Best RMSE model: Random Forest
Best R² model: Random Forest


# **ML2 for SPI**

In [471]:
ridge.fit(X2_train, y2_train)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [472]:
rf.fit(X2_train, y2_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [473]:
gb.fit(X2_train, y2_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf

In [474]:
y2_pred_ridge = ridge.predict(X2_test)
y2_pred_rf = rf.predict(X2_test)
y2_pred_gb = gb.predict(X2_test)

In [475]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ridge_mae = mean_absolute_error(y2_test, y2_pred_ridge)
ridge_rmse = np.sqrt(mean_squared_error(y2_test, y2_pred_ridge))
ridge_r2 = r2_score(y2_test, y2_pred_ridge)

In [476]:
rf_mae = mean_absolute_error(y2_test, y2_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y2_test, y2_pred_rf))
rf_r2 = r2_score(y2_test, y2_pred_rf)

In [477]:
gb_mae = mean_absolute_error(y2_test, y2_pred_gb)
gb_rmse = np.sqrt(mean_squared_error(y2_test, y2_pred_gb))
gb_r2 = r2_score(y2_test, y2_pred_gb)

In [478]:
results_sqli = pd.DataFrame({
    "Model": ["Ridge", "Random Forest", "Gradient Boosting"],
    "MAE": [ridge_mae, rf_mae, gb_mae],
    "RMSE": [ridge_rmse, rf_rmse, gb_rmse],
    "R²": [ridge_r2, rf_r2, gb_r2]
})

results_sqli.round(3)

,Model,MAE,RMSE,R²
0,Ridge,1.951,2.560,0.956
1,Random Forest,5.342,6.778,0.691
2,Gradient Boosting,3.657,5.393,0.804


In [479]:
best_mae_model = results_sqli.loc[results_sqli["MAE"].idxmin(), "Model"]
best_rmse_model = results_sqli.loc[results_sqli["RMSE"].idxmin(), "Model"]
best_r2_model = results_sqli.loc[results_sqli["R²"].idxmax(), "Model"]

print("Best MAE model:", best_mae_model)
print("Best RMSE model:", best_rmse_model)
print("Best R² model:", best_r2_model)

Best MAE model: Ridge
Best RMSE model: Ridge
Best R² model: Ridge
